# 1. Hyperparameter Optimization & Residual Diagnostics

## 2. Objective
Tune ensemble regressors using temporal cross-validation (`TimeSeriesSplit`) without lookahead bias. Perform detailed residual diagnostics.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

processed_path = "../datasets/processed/traffic_processed.csv"
df = pd.read_csv(processed_path)
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values('date_time').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df, test_df = df.iloc[:split_idx], df.iloc[split_idx:]

features = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main']
target = 'traffic_volume'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

numeric_features = ['hour', 'day_of_week', 'month', 'temp', 'rain_1h', 'snow_1h', 'clouds_all']
categorical_features = ['weather_main']
binary_features = ['is_weekend', 'is_rush_hour']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('passthrough', 'passthrough', binary_features)
    ]
)
print("Data split and preprocessor ready.")

## 3. Time-Series Cross Validation & Grid Search

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('gbr', GradientBoostingRegressor(random_state=42))
])

param_grid = {
    'gbr__n_estimators': [100, 150],
    'gbr__learning_rate': [0.08, 0.12],
    'gbr__max_depth': [5, 6]
}

print("Running TimeSeriesSplit GridSearchCV...")
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print(f"Optimal Hyperparameters: {grid_search.best_params_}")
print(f"Best Time-Series CV R²: {grid_search.best_score_:.4f}")

## 4. Final Evaluation and Residual Diagnostics

In [ ]:
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Tuned Model Performance on Unseen Test Split:")
print(f"MAE:  {mae:.2f} vehicles/hour")
print(f"RMSE: {rmse:.2f} vehicles/hour")
print(f"R²:   {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.2, color='#2b5c8f', s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title("Actual vs. Predicted Traffic Volume", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Actual Volume (vehicles/hr)")
axes[0].set_ylabel("Predicted Volume (vehicles/hr)")

# Residual Distribution
sns.histplot(residuals, kde=True, ax=axes[1], color='#e74c3c', bins=40)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title("Residual Error Distribution (Zero-Centered)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()